In [1]:
import pandas as pd
from collections import Counter, defaultdict
import numbers
import numpy as np
import fitz
from pathlib import Path

dev_set_base_dir = "path/to/dev-set-100"
dev_set_wald_wvc_base_dir = "path/to/dev-set-Wald-WVC"


In [2]:
df = pd.read_json("../data/interim/faktencheck-db/faktenscheck_core_corrected.jsonl", lines=True)
print(df.head())

  zotitem_ptr_id biodiversity_level  \
0       25ABQZIH               None   
1       25RIYD2C               None   
2       29Q84X7V               None   
3       2E9XWUUE               None   
4       2EUNPHDZ               None   

                                      ecosystem_type  \
0  [{'category': 'III Terrestrische und semiterre...   
1  [{'category': 'III Terrestrische und semiterre...   
2  [{'category': 'III Terrestrische und semiterre...   
3  [{'category': 'I Biotoptypengruppen der Meere ...   
4  [{'category': 'I Biotoptypengruppen der Meere ...   

                     habitat  \
0       Agrar- und Offenland   
1       Agrar- und Offenland   
2       Agrar- und Offenland   
3  Küsten und Küstengewässer   
4  Küsten und Küstengewässer   

                                                taxa  
0  [{'species_group': 'Insekten'}, {'species_grou...  
1                                               None  
2                                               None  
3  [{'species_g

In [3]:
counters = defaultdict(Counter)
totals = defaultdict(int)
numeric_sums = defaultdict(float)


def process(col_name, value):
    """Flattens lists/dicts and updates counters."""

    # dict → rekursiv flatten
    if isinstance(value, dict):
        for k, v in value.items():
            process(f"{col_name}.{k}", v)
        return

    # list → jedes Element einzeln
    if isinstance(value, list):
        for item in value:
            process(col_name, item)
        return

    # primitive value
    counters[col_name][value] += 1
    totals[col_name] += 1

    if isinstance(value, numbers.Number):
        numeric_sums[col_name] += value


for col in df.columns:
    for value in df[col].dropna():
        process(col, value)


# Output
for col in sorted(counters):
    print(f"\n=== {col} ===")
    print(f"Total values: {totals[col]}")

    if numeric_sums[col] != 0:
        print(f"Numeric sum: {numeric_sums[col]}")

    #for val, count in counters[col].most_common():
    #    print(f"  {val}: {count}")


=== biodiversity_level ===
Total values: 132

=== ecosystem_type.category ===
Total values: 251

=== ecosystem_type.term ===
Total values: 251

=== habitat ===
Total values: 190

=== taxa.species_group ===
Total values: 215

=== zotitem_ptr_id ===
Total values: 100


In [4]:
df = pd.read_csv("../data/external/organism_trends/Weighted Vote Count Wald Literatur - Sheet1.csv")
print(df.head())

     N          Key Bemerkung_Maria    Autor  Jahr  \
0  214  Schmidt2012             NaN  Schmidt  2012   
1  215  Schmidt2012             NaN  Schmidt  2012   
2  216  Schmidt2012             NaN  Schmidt  2012   
3  217  Schmidt2012             NaN  Schmidt  2012   
4  218  Schmidt2012             NaN  Schmidt  2012   

                                               Titel Zeitschrift  DOI  \
0  13 Jahre nach dem Sturm - Vegetationsentwicklu...    Hercynia  NaN   
1  13 Jahre nach dem Sturm - Vegetationsentwicklu...    Hercynia  NaN   
2  13 Jahre nach dem Sturm - Vegetationsentwicklu...    Hercynia  NaN   
3  13 Jahre nach dem Sturm - Vegetationsentwicklu...    Hercynia  NaN   
4  13 Jahre nach dem Sturm - Vegetationsentwicklu...    Hercynia  NaN   

   ID_Studie            Screener  ... Massnahme_Schutz  \
0        NaN  Emilia Schlotfeldt  ...              NaN   
1        NaN  Emilia Schlotfeldt  ...              NaN   
2        NaN  Emilia Schlotfeldt  ...              NaN   
3   

In [5]:
counters = defaultdict(Counter)
totals = defaultdict(int)
numeric_sums = defaultdict(float)
for col in df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]].columns:
    for value in df[col].dropna():
        process(col, value)


# Output
for col in sorted(counters):
    print(f"\n=== {col} ===")
    print(f"Total values: {totals[col]}")

    if numeric_sums[col] != 0:
        print(f"Numeric sum: {numeric_sums[col]}")

    for val, count in counters[col].most_common():
        print(f"  {val}: {count}")


=== Antwortvariable ===
Total values: 315
  Abundanz: 191
  Artenzahl: 107
  ENS: 17

=== Hauptgruppe_RoteListen ===
Total values: 315
  Pflanzen: 144
  Wirbeltiere: 87
  Wirbellose: 65
  Pilze_Flechten: 19

=== Lebensraum ===
Total values: 315
  Wald: 315

=== Trend ===
Total values: 315
  positive: 111
  negative: 85
  no: 75
  negative to positive: 30
  positive to negative: 14


In [12]:
df = pd.read_json("../data/processed/faktencheck/dev-set-100/predictions.jsonl", lines=True)
print(df.head())

      file_name                                               text  \
0  25ABQZIH.pdf  # Informationspapier des Greifswald Moor Centr...   
1  25RIYD2C.pdf  27.08.25, 10:02 Grünes Band Deutschland: Chron...   
2  29Q84X7V.pdf  ## **WHAT WE CAN LEARN FROM THE GERMAN** **IMP...   
3  2E9XWUUE.pdf  # Neobiota der deutschen Nord- und Ostseeküste...   
4  2EUNPHDZ.pdf  [See discussions, stats, and author profiles f...   

   response_content  structured  structured_with_metadata  reasoning_content  \
0               NaN         NaN                       NaN                NaN   
1               NaN         NaN                       NaN                NaN   
2               NaN         NaN                       NaN                NaN   
3               NaN         NaN                       NaN                NaN   
4               NaN         NaN                       NaN                NaN   

   messages  messages_formatted errors errors_long  
0       NaN                 NaN     []       

In [16]:
def count_words(df, col):
    word_counts = []
    
    for value in df[col].dropna():
        if not isinstance(value, str):
            continue
    
        words = value.split()   # whitespace split
        word_counts.append(len(words))
    
    # Summary stats
    sum_words = np.sum(word_counts) if word_counts else 0
    avg_words = np.mean(word_counts) if word_counts else 0
    med_words = np.median(word_counts) if word_counts else 0
    max_words = max(word_counts) if word_counts else 0
    min_words = min(word_counts) if word_counts else 0
    
    print(f"Column: {col}")
    print(f"Average words per doc: {avg_words:.2f}")
    print(f"Median words per doc: {med_words}")
    print(f"Max words per doc: {max_words}")
    print(f"Min words per doc: {min_words}")
    print(f"Total words: {sum_words}")


col = "text"
count_words(df, col)


Column: text
Average words per doc: 37764.40
Median words per doc: 9388.0
Max words per doc: 624812
Min words per doc: 0
Total words: 3776440


In [17]:
def count_pages(df, col, base_dir):
    page_counts = []
    
    for file_path in df[col].dropna().unique():
        try:
            doc = fitz.open(Path(base_dir, file_path))
            num_pages = doc.page_count
            page_counts.append(num_pages)
            doc.close()
    
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    sum_pages = np.sum(page_counts) if page_counts else 0
    avg_pages = np.mean(page_counts) if page_counts else 0
    med_pages = np.median(page_counts) if page_counts else 0
    max_pages = max(page_counts) if page_counts else 0
    min_pages = min(page_counts) if page_counts else 0
    
    print(f"Average pages per PDF: {avg_pages:.2f}")
    print(f"Median pages per PDF: {med_pages}")
    print(f"Max pages: {max_pages}")
    print(f"Min pages: {min_pages}")
    print(f"Total pages: {sum_pages}")

col = "file_name"
count_pages(df, col, dev_set_base_dir)

Average pages per PDF: 96.09
Median pages per PDF: 16.0
Max pages: 2218
Min pages: 2
Total pages: 9609


In [18]:
df = pd.read_json("../data/processed/faktencheck/dev-set-Wald-WVC/predictions.jsonl.gz", lines=True)
print(df.head())

      file_name                                               text  \
0  26QI8J7B.pdf  Ecological Complexity 7 (2010) 260–272\n\n\n[C...   
1  2AWXR7KG.pdf                                                      
2  2P53UVJA.pdf  ### **Invasion Ecology**\n\n\n# **Invasion Eco...   
3  2SNIVXFW.pdf  This is a pre-print version of the following p...   
4  2US2J9GI.pdf  Ecological Applications, 22(8), 2012, pp. 2065...   

   response_content  structured  structured_with_metadata  reasoning_content  \
0               NaN         NaN                       NaN                NaN   
1               NaN         NaN                       NaN                NaN   
2               NaN         NaN                       NaN                NaN   
3               NaN         NaN                       NaN                NaN   
4               NaN         NaN                       NaN                NaN   

   messages  messages_formatted errors errors_long  
0       NaN                 NaN     []       

In [19]:
col = "text"
count_words(df, col)

Column: text
Average words per doc: 14193.14
Median words per doc: 9045.0
Max words per doc: 519607
Min words per doc: 0
Total words: 5804996


In [11]:
col = "file_name"
count_pages(df, col, dev_set_wald_wvc_base_dir)

Average pages per PDF: 30.48
Median pages per PDF: 14.0
Max pages: 894
Min pages: 1
Total pages: 12468
